# ML-08 — Capstone Modeling Lane

**Scope: content-refresh prioritization model (W05), compared to the Week-4 baseline.**

This notebook trains a logistic regression prioritization model and compares it with the Week-4 rule baseline
on the **same client-grouped test split** and the **same precision@K metric**. It runs top to bottom.

*For the ML-08 assignment: read `skills/README.md` first; load `training-honest-models` and `flyrank-data`.*

## 1. Method choice and why

**My lane is prioritization**: the business question is "which pages should I refresh first?", not a yes/no audit. The W04 baseline already builds a ranked queue (action label `REFRESH_CONTENT`), so the learned model must produce the same kind of output: a ranking score.

**Chosen method: Logistic Regression** — its predicted probability `P(decline_target = 1)` is used as the ranking score.

Why it fits this lane:

1. The observed outcome is **binary** (`decline_target = 1` when `trend_direction == "down"`, else 0).
2. It returns a **probability**, a natural ranking score — the sorted test rows give the refresh queue.
3. It is **interpretable**: one coefficient per feature that a human can read and sanity-check.
4. It is **simple** — 24 carefully-chosen features, no fancy ensembling. The assignment says simplicity is a feature; I only add complexity if the baseline comparison earns it.

### Leakage guard

- the ranking target is `decline_target` (from `trend_direction`).
- `trend_direction` / `trend_pct` are **never** model features.
- `content_id` / `client_id` are **never** model features — `client_id` is used only for the group split.

## 2. Split design

**Design: client-grouped 80/20 train/test split** (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`, groups=`client_id`).

Pages from the same client share keyword logic, publishing rhythm and measurement quirks — they are correlated. A plain random split would let the same client appear in both train and test, which flatters the metrics but doesn't reflect how the model will be used on **new clients**. Grouping on `client_id` puts every client on exactly one side, so the test measures performance on clients never seen in training — the honest check for a decision-support tool.

Honesty details:

- seeds fixed (`random_state=42`) for split and model; library versions printed for reproducibility.
- Preprocessing (median/most-frequent imputation, scaling, one-hot) is fitted **on the training folds only**, then the test set is transformed through the fitted pipeline — no test information reaches fit time.
- Code below asserts the train/test client sets are disjoint and that no forbidden column is used as a feature.

In [1]:
# Import required libraries
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

# Reproducibility note: all randomness uses random_state=42.
print("Library versions (for reproducibility):")
print(f"  python        : {sys.version.split()[0]}")
print(f"  pandas        : {pd.__version__}")
print(f"  numpy         : {np.__version__}")
print(f"  scikit-learn  : {sklearn.__version__}")

Library versions (for reproducibility):
  python        : 3.13.0
  pandas        : 2.2.3
  numpy         : 2.1.2
  scikit-learn  : 1.5.2


In [2]:
# Load dataset
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Distinct clients:", df["client_id"].nunique())

Dataset shape: (30000, 44)
Distinct clients: 32


In [3]:
# Define the observed target.
# The dataset has no pre-made label, so we define it the same way the prep step
# would: decline_target = 1 when trend_direction == "down", else 0.
# trend_direction is used ONLY here to build the label - NEVER as a feature.
df["decline_target"] = (df["trend_direction"] == "down").astype(int)

print("decline_target counts:")
print(df["decline_target"].value_counts().sort_index())
print("observed decline rate: %.4f" % df["decline_target"].mean())

decline_target counts:
decline_target
0    13738
1    16262
Name: count, dtype: int64
observed decline rate: 0.5421


In [4]:
# Define leakage-safe features. The 24 features from the assignment card.
CATEGORICAL_FEATURES = ["content_type", "main_intent"]

NUMERIC_FEATURES = [
    # keyword context
    "search_volume", "competition", "cpc",
    # content properties
    "word_count", "char_count",
    # 90-day activity
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    # recency
    "content_age_days", "days_since_last_update",
    # derived rates
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
assert len(FEATURES) == 24 and len(set(FEATURES)) == 24, "feature list is not 24 unique columns"

# forbid identifiers and everything the label is derived from
forbidden = {"content_id", "client_id", "trend_direction", "trend_pct"}
assert set(FEATURES).isdisjoint(forbidden), "a leaked column is in FEATURES"

print(f"Selected features: {len(FEATURES)}  ({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")
print("Missing values in selected features (rows):")
miss = df[FEATURES].isna().sum()
print(miss[miss > 0])

Selected features: 24  (22 numeric, 2 categorical)
Missing values in selected features (rows):
search_volume    2468
competition      2468
cpc              2468
word_count       7699
char_count       7699
scroll_rate       125
main_intent      2374
dtype: int64


In [5]:
# Why explicit imputation instead of fillna(0): missingness follows content_type.
miss_vol = (
    df.groupby("content_type")["search_volume"]
      .apply(lambda s: round(s.isna().mean(), 3))
      .rename("search_volume missing rate")
)
print(miss_vol)

content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume missing rate, dtype: float64


In [6]:
# Create client-grouped train/test split.
groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, df["decline_target"], groups=groups))

X_train = df.loc[train_idx, FEATURES]
X_test  = df.loc[test_idx,  FEATURES]
y_train = df.loc[train_idx, "decline_target"]
y_test  = df.loc[test_idx,  "decline_target"]
clients_train = df.loc[train_idx, "client_id"]
clients_test  = df.loc[test_idx,  "client_id"]

print(f"Train rows: {len(X_train)}   Test rows: {len(X_test)}")
print(f"Train clients: {clients_train.nunique()}   Test clients: {clients_test.nunique()}")
print(f"Train target rate: {y_train.mean():.4f}   Test target rate: {y_test.mean():.4f}")

Train rows: 23837   Test rows: 6163
Train clients: 25   Test clients: 7
Train target rate: 0.5501   Test target rate: 0.5110


In [7]:
# Verify client separation: no client appears in BOTH train and test.
train_set, test_set = set(clients_train), set(clients_test)
overlap = train_set & test_set

print("Clients in both train and test:", len(overlap))
assert len(overlap) == 0, "A client appears in BOTH sides - split is NOT honest!"
print("Clients only in train:", len(train_set - test_set))
print("Clients only in test :", len(test_set - train_set))
print("VERIFIED: train and test client sets are disjoint.")

Clients in both train and test: 0
Clients only in train: 25
Clients only in test : 7
VERIFIED: train and test client sets are disjoint.


## 3. Train + compare vs my baseline

**Same data, same split, same metric.** The model and the Week-4 baseline are evaluated on the *same* 20% test rows with the *same* metric: `precision@K` on a ranked list (`baseline_score`/probability descending). The baseline is recomputed **in this notebook, on the test rows** rather than reusing a previously exported file.

### The Week-4 baseline rule (recomputed here)

- `stale` = `days_since_last_update >= 180`
- `visible` = `impressions_90d >= 500`
- `position_opportunity` = `avg_position >= 10`
- `baseline_score = stale + visible + position_opportunity`
- baseline queue sorted by `baseline_score` desc, then `impressions_90d` desc (same tie-break as W04).

In [8]:
# Build a leakage-safe preprocessing + model pipeline.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

# Logistic Regression: primary learned model (see Section 1).
logreg = LogisticRegression(random_state=42, max_iter=1000)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("logreg", logreg),
])

# Fit the whole pipeline (imputers + scaler + one-hot + classifier) on TRAIN ONLY.
model.fit(X_train, y_train)
print("Pipeline fitted on training rows only.")

Pipeline fitted on training rows only.


In [9]:
# Generate ranking scores on the test set.
y_pred_proba = model.predict_proba(X_test)[:, 1]   # P(observed decline == 1)

print("Predicted probabilities (test):")
print(pd.Series(y_pred_proba, name="P(decline)").describe().round(4))

Predicted probabilities (test):
count    6163.0000
mean        0.5268
std         0.1579
min         0.0661
25%         0.4293
50%         0.5255
75%         0.6399
max         0.9216
Name: P(decline), dtype: float64


In [10]:
# Evaluation: ranking-oriented metrics. K = 20, 50, 100.
def precision_at_k(y_true, scores, k):
    # Fraction of 'decline' rows among the top-k by scores (descending).
    assert k <= len(y_true), f"K={k} exceeds the {len(y_true)} test rows"
    order = np.argsort(-np.asarray(scores), kind="stable")
    top_k = np.asarray(y_true, dtype=float)[order[:k]]
    return float(top_k.mean())

KS = [20, 50, 100]

# ---- MODEL metrics ----
model_prec = {k: precision_at_k(y_test, y_pred_proba, k) for k in KS}
base_rate  = float(y_test.mean())

print("Logistic Regression (test set):")
for k in KS:
    print(f"  precision@{k:<3}: {model_prec[k]:.4f}")
print(f"  base rate      : {base_rate:.4f}")
print(f"  ROC-AUC        : {roc_auc_score(y_test, y_pred_proba):.4f}")

Logistic Regression (test set):
  precision@20 : 0.8000
  precision@50 : 0.7200
  precision@100: 0.6700
  base rate      : 0.5110
  ROC-AUC        : 0.5964


In [11]:
# ---- Recreate the WEEK-4 baseline ON THE TEST ROWS (not from a file) ----
test_df = df.loc[test_idx].copy()
test_df["stale"]  = (test_df["days_since_last_update"] >= 180).astype(int)
test_df["visible"] = (test_df["impressions_90d"] >= 500).astype(int)
test_df["position_opportunity"] = (test_df["avg_position"] >= 10).astype(int)
test_df["baseline_score"] = (
    test_df["stale"] + test_df["visible"] + test_df["position_opportunity"]
)

print("Week-4 baseline_score on the test rows:")
print(test_df["baseline_score"].value_counts().sort_index())

Week-4 baseline_score on the test rows:
baseline_score
0    1734
1    2849
2    1580
Name: count, dtype: int64


In [12]:
# ---- BASELINE metrics on the same rows ----
baseline_ranked = test_df.sort_values(
    by=["baseline_score", "impressions_90d"],
    ascending=[False, False],
)
baseline_scores = baseline_ranked["baseline_score"].to_numpy()
baseline_labels = baseline_ranked["decline_target"].to_numpy()

base_prec = {k: precision_at_k(baseline_labels, baseline_scores, k) for k in KS}

print("Week-4 baseline (same test set):")
for k in KS:
    print(f"  precision@{k:<3}: {base_prec[k]:.4f}")
print(f"  base rate          : {baseline_labels.mean():.4f}")
print(f"  ROC-AUC (on score) : {roc_auc_score(y_test, baseline_scores):.4f}"
      "     (score is only 4 discrete levels, so approximate)")

Week-4 baseline (same test set):
  precision@20 : 0.4500
  precision@50 : 0.4200
  precision@100: 0.4300
  base rate          : 0.5110
  ROC-AUC (on score) : 0.4918     (score is only 4 discrete levels, so approximate)


In [13]:
# ---- Final comparison table (same rows, same metric) ----
comparison = pd.DataFrame({
    "Method": ["Week-4 baseline", "Logistic Regression"],
    "Precision@20":  [base_prec[20],  model_prec[20]],
    "Precision@50":  [base_prec[50],  model_prec[50]],
    "Precision@100": [base_prec[100], model_prec[100]],
    "Base rate":     [baseline_labels.mean(), base_rate],
}).set_index("Method")
comparison

,Precision@20,Precision@50,Precision@100,Base rate
Method,,,,
Week-4 baseline,0.45,0.42,0.43,0.510952
Logistic Regression,0.80,0.72,0.67,0.510952


### How to read the table

Every number is computed on the **same client-disjoint 20% test set** with the **same precision@K metric** (fraction of the top-K rows whose observed outcome is `decline_target == 1`). The **test base rate ≈ 0.51** is what a random ranker would achieve at every K, so both methods should sit clearly above it to be useful.

Observed results on this one split — the wording I use throughout is directional, not causal.

## 4. Errors and interpretation

In [14]:
# Inspect model coefficients, mapped back to readable feature names.
onehot = model.named_steps["preprocessing"].named_transformers_["cat"].named_steps["onehot"]
cat_names = list(onehot.get_feature_names_out(CATEGORICAL_FEATURES))
all_features = NUMERIC_FEATURES + cat_names

coefs = pd.DataFrame({
    "feature":     all_features,
    "coefficient": model.named_steps["logreg"].coef_.ravel(),
})
coefs["abs_coef"] = coefs["coefficient"].abs()
coefs = coefs.sort_values("abs_coef", ascending=False).drop(columns=["abs_coef"]).reset_index(drop=True)

print("Top positive coefficients  (larger values -> higher predicted 'decline'):")
print(coefs[coefs["coefficient"] > 0].head(10).to_string(index=False))
print()
print("Top negative coefficients  (larger values -> higher predicted 'not decline'):")
print(coefs[coefs["coefficient"] < 0].head(10).to_string(index=False))

Top positive coefficients  (larger values -> higher predicted 'decline'):
                     feature  coefficient
                sessions_90d     0.777865
       days_with_impressions     0.602827
content_type_keyword article     0.340541
           scroll_events_90d     0.274967
                  word_count     0.230450
   main_intent_informational     0.215671
               pageviews_90d     0.194551
                 scroll_rate     0.162938
      days_since_last_update     0.158611
      main_intent_commercial     0.110399

Top negative coefficients  (larger values -> higher predicted 'not decline'):
                        feature  coefficient
                      users_90d    -0.992763
       main_intent_navigational    -0.506100
             days_with_sessions    -0.425024
               content_age_days    -0.399683
    content_type_feedly article    -0.387148
                     char_count    -0.196771
                   avg_position    -0.146787
content_type_comparison a

**Coefficients are associations, not causes.** A positive weight raises the predicted probability of `decline` as the feature value grows; a negative weight lowers it. Numeric features are z-scored (standardized) so their weights are directly comparable; categorical one-hot weights compare each level with the encoder's baseline level. With `avg_position`, position is *lower is better* — interpret signs relative to that standard.

In [15]:
# Error analysis: false positives (predicted high, observed not declining) vs false negatives.
analysis = test_df.copy()
analysis["pred_prob"] = y_pred_proba
analysis["pred_label"] = (analysis["pred_prob"] >= 0.5).astype(int)

fp = analysis[(analysis["pred_label"] == 1) & (analysis["decline_target"] == 0)].sort_values("pred_prob", ascending=False)
fn = analysis[(analysis["pred_label"] == 0) & (analysis["decline_target"] == 1)].sort_values("pred_prob", ascending=True)

print(f"False positives (model says 'down', observed NOT down): {len(fp)}")
print(f"False negatives (model says 'ok',  observed down)    : {len(fn)}")

# anonymized display columns - no content ids, no client ids, no URLs.
SHOW_COLS = [
    "content_type", "main_intent", "word_count", "impressions_90d",
    "content_age_days", "days_since_last_update", "avg_position", "ctr",
    "pred_prob", "decline_target",
]

False positives (model says 'down', observed NOT down): 1503
False negatives (model says 'ok',  observed down)    : 1149


In [16]:
# Three concrete false-positive cases (highest confidence, wrong).
print("False-positive examples (high predicted risk, observed NOT down):")
fp[SHOW_COLS].head(3)

False-positive examples (high predicted risk, observed NOT down):


,content_type,main_intent,word_count,impressions_90d,content_age_days,days_since_last_update,avg_position,ctr,pred_prob,decline_target
10175,keyword article,commercial,2675.0,235,181,20,31.0,0.85,0.907084,0
8016,keyword article,informational,2737.0,2164,95,20,8.1,0.23,0.895309,0
26614,keyword article,informational,2481.0,290,96,20,5.9,0.00,0.888293,0


In [17]:
# Three concrete false-negative cases (predicted safe, observed down).
print("False-negative examples (low predicted risk, observed down):")
fn[SHOW_COLS].head(3)

False-negative examples (low predicted risk, observed down):


,content_type,main_intent,word_count,impressions_90d,content_age_days,days_since_last_update,avg_position,ctr,pred_prob,decline_target
17127,keyword article,informational,NaN,83603,487,104,3.4,1.06,0.066095,1
1010,keyword article,transactional,NaN,21103,460,22,4.2,1.11,0.114104,1
26413,keyword article,informational,3033.0,64718,421,14,3.1,0.84,0.122490,1


In [18]:
# Do errors cluster? Compare FP / FN against all test rows on key feature ranges.
def cohort(rows, name):
    return {
        "cohort": name,
        "n": len(rows),
        "missing word_count share": round(rows["word_count"].isna().mean(), 3),
        "median content_age_days": round(rows["content_age_days"].median()),
        "median impressions_90d": round(rows["impressions_90d"].median()),
        "avg_position==0 share": round((rows["avg_position"] == 0).mean(), 3),
        "median days_since_last_update": round(rows["days_since_last_update"].median()),
        "share keyword article": round((rows["content_type"] == "keyword article").mean(), 3),
    }

clusters = pd.DataFrame([
    cohort(analysis, "all test"),
    cohort(fp, "false positives"),
    cohort(fn, "false negatives"),
]).set_index("cohort")
clusters

,n,missing word_count share,median content_age_days,median impressions_90d,avg_position==0 share,median days_since_last_update,share keyword article
cohort,,,,,,,
all test,6163,0.176,277,429,0.010,20,1.0
false positives,1503,0.098,237,874,0.012,20,1.0
false negatives,1149,0.283,390,68,0.000,20,1.0


### Where the model is most wrong

Comparing the cohort numbers above (`all test`, `false positives`, `false negatives`):

- **False positives — 1,503 rows** (predicted high, observed *not* down): the model leans hardest on the **activity side**. These pages have *higher* impressions (median 874 vs 429 across all test) and are *younger* (median `content_age_days` 237 vs 277). They are visible, higher-signal pages that the model flags hard even though they are not observed sliding in this window.
- **False negatives — 1,149 rows** (predicted safe, observed down): these are the **hygiene pages**. Older (median `content_age_days` 390 vs 277), low-impression (median 68 vs 429), with **more missing word-count data** (0.28 vs 0.18), and every row has position data (`avg_position == 0` share 0.00 — the three cases shown sit at positions ~3–4). These are exactly the pages the model's "not declining" weights cover.
- **Content-type caveat:** each cohort here is observed to be 100% *keyword article* — the seven test clients only publish keyword articles in this slice, so the split does not exercise the feedly/comparison missing-data pattern. A follow-up split that places feedly/comparison clients in the test set is a sensible next check.

Observed error groupings, used as input for the next iteration — not causal claims.

### Baseline vs. model — final look

Re-run the takeaway table here on purpose:

In [19]:
comparison

,Precision@20,Precision@50,Precision@100,Base rate
Method,,,,
Week-4 baseline,0.45,0.42,0.43,0.510952
Logistic Regression,0.80,0.72,0.67,0.510952


### Baseline vs. model — verdict (5 sentences)

On this client-grouped test slice the model beats the Week-4 baseline at every K — Precision@20 **0.80 vs 0.45**, Precision@50 **0.72 vs 0.42**, Precision@100 **0.67 vs 0.43** — against a test base rate of **0.51**. The lift is concentrated at the head of the ranking (ROC-AUC only 0.60), so the honest description is "an improved queue-orderer", not a globally better classifier. In this fold the baseline's top pages are score-2 visibility/opportunity rows and only ~0.43 of them were observed to be declining, so the rule's ranking premise simply did not transfer to these held-out clients. The coefficients are interpretable (higher session activity, days-with-impressions and keyword-articles push predicted decline; higher user counts, navigational intent and comparison articles pull it down), with one counter-intuitive sign worth a sanity check: the model gives `avg_position` a *negative* weight, the opposite of the baseline's position-opportunity assumption. Given the modest AUC and the keyword-only test fold, I recommend using the model as **decision-support** for the refresh queue, not an automatic decision-maker, until a follow-up split that includes feedly/comparison clients confirms the gain.

## Self-check

- [x] Every section covered — markdown thinking plus the code that backs it
- [x] Notebook runs **top to bottom with no errors** (verified by running it)
- [x] No client names, URLs, or private queries anywhere in this file
- [x] Claims use careful language: **observed, measured, directional, decision-support**
- [x] Model and Week-4 baseline compared on the **same test split**, **same metric** (precision@K), **same rows**
- [x] Preprocessing fit on **training only** (imputer / scaler / one-hot inside the Pipeline)
- [x] `content_id`, `client_id`, `trend_direction`, `trend_pct` never used as predictive features
- [x] Reproducible: `random_state=42`, deterministic split, library versions printed
- [x] Saved as `work/notebooks/w05_model.ipynb`